**生成式 AI 使用说明**：本作业中使用生成式 AI 工具时，适用与协作相同的课程政策。和其他协作者一样，每位学生都必须独立写出自己的解答，不能直接依赖交互输出；提交内容中还应注明协作的性质。使用生成式 AI 工具实质性完成作业内容不符合本作业的精神，也会违反 [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# 将你的 Google Drive 挂载到 Colab VM。
from google.colab import drive
drive.mount('/content/drive')

# TODO：填写 Drive 中保存解压后
# 作业文件夹，例如 'cs231n/assignments/assignment3/'
FOLDERNAME = "cs231n/assignments/assignment3/"
assert FOLDERNAME is not None, "[!] Enter the foldername."

# 现在已经挂载 Drive，下面确保
# Colab VM 的 Python 解释器可以加载
# 其中的 Python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# 将 COCO 数据集下载到你的 Drive
# 如果它还不存在。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

## 使用 GPU

进入 `Runtime > Change runtime type`，将 `Hardware accelerator` 设置为 `GPU`。这会重置 Colab。**请重新运行顶部单元，再次挂载你的 Drive。**

# 自监督学习

## 什么是自监督学习？
现代机器学习需要大量标注数据。但很多时候，获取大量人工标注数据既困难又昂贵。有没有办法让机器在没有标注数据集的情况下，自动学习一个能生成良好视觉表示的模型？有，这就是自监督学习。

自监督学习（SSL）允许模型在不使用标签的情况下，仅利用给定数据集中的数据自动学习一个“好”的表示空间。具体来说，如果我们的数据集是一组图像，自监督学习就能让模型为图像学习并生成“好”的表示向量。

SSL 方法近年来越来越受欢迎，原因是学到的模型在其他数据集上也能继续表现良好，也就是模型没有训练过的新数据集上也能迁移。

## 什么是“好”的表示？
一个“好”的表示向量需要捕捉图像中与整个数据集相关的重要特征。这意味着，数据集中表示语义相似实体的图像应该有相似的表示向量，而不同图像应该有不同的表示向量。例如，两张苹果图像应有相似表示向量，而苹果图像和香蕉图像应有不同表示向量。

## 对比学习：SimCLR
最近，[SimCLR](https://arxiv.org/pdf/2002.05709.pdf) 提出了一种使用**对比学习**来学习良好视觉表示的新架构。对比学习旨在为相似图像学习相似表示，为不同图像学习不同表示。正如我们将在这个 notebook 中看到的，这个简单想法让我们无需任何标签也能训练出相当不错的模型。

具体来说，对数据集中的每张图像，SimCLR 会生成该图像的两个不同增强视图，称为**正样本对**。然后，模型被鼓励为这一对图像生成相似的表示向量。下面是该架构的示意图（来自论文 Figure 2）。

In [ ]:
# 运行这个 cell 查看 SimCLR architecture。
from IPython.display import Image
Image('images/simclr_fig2.png', width=500)

给定一张图像 **x**，SimCLR 使用两种不同的数据增强方案 **t** 和 **t'** 生成正样本对图像 **$\tilde{x}_i$** 和 **$\tilde{x}_j$**。$f$ 是一个基础 encoder network，它从增强后的数据样本中提取表示向量，分别得到 **$h_i$** 和 **$h_j$**。最后，一个小型神经网络 projection head $g$ 会把表示向量映射到应用对比损失的空间。对比损失的目标是最大化最终向量 **$z_i = g(h_i)$** 和 **$z_j = g(h_j)$** 之间的一致性。稍后我们会更详细讨论对比损失，并由你来实现它。

训练完成后，我们丢弃 projection head $g$，只使用 $f$ 和表示 $h$ 执行下游任务，例如分类。你会有机会在训练好的 SimCLR 模型之上微调一个层来完成分类任务，并把它的性能与 baseline 模型（没有自监督学习）进行比较。

## 预训练权重
为方便你完成作业，我们提供了 SimCLR 模型的预训练权重（在 CIFAR-10 上训练约 18 小时）。运行下面的单元下载稍后会用到的预训练模型权重。（大约需要 1 分钟）

In [ ]:
%%bash
DIR=pretrained_model/
if [ ! -d "$DIR" ]; then
    mkdir "$DIR"
fi

URL=http://downloads.cs.stanford.edu/downloads/cs231n/pretrained_simclr_model.pth
# 如果上面的 URL 不可用，尝试这个。
# URL=http://cs231n.stanford.edu/2025/storage/a3/pretrained_simclr_model.pth
FILE=pretrained_model/pretrained_simclr_model.pth
if [ ! -f "$FILE" ]; then
    echo "Downloading weights..."
    wget "$URL" -O "$FILE"
fi

In [ ]:
# 设置单元。
%pip install thop
import torch
import os
import sys
import types
import importlib
import pandas as pd
import numpy as np
import torch.optim as optim
import torch.nn as nn
import random
from thop import profile, clever_format
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
import matplotlib.pyplot as plt
%matplotlib inline

if "imp" not in sys.modules:
    imp = types.ModuleType("imp")
    imp.reload = importlib.reload
    sys.modules["imp"] = imp

%load_ext autoreload
%autoreload 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 数据增强

第一步是执行数据增强。在 `cs231n/simclr/data_utils.py` 中实现 `compute_train_transform()` 函数，应用下列随机变换：

1. 随机 resize 并裁剪到 `32x32`。
2. 以 0.5 的概率水平翻转图像。
3. 以 0.8 的概率应用 color jitter（定义见 `compute_train_transform()`）。
4. 以 0.2 的概率把图像转换为灰度。

现在完成 `cs231n/simclr/data_utils.py` 中的 `compute_train_transform()` 和 `CIFAR10Pair.__getitem__()`，应用数据增强变换，并生成 **$\tilde{x}_i$** 和 **$\tilde{x}_j$**。

测试并确认你的数据增强代码正确：

In [ ]:
from cs231n.simclr.data_utils import *
from cs231n.simclr.contrastive_loss import *

answers = torch.load('simclr_sanity_check.key')

In [ ]:
from PIL import Image
import torchvision
from torchvision.datasets import CIFAR10

def test_data_augmentation(correct_output=None):
    train_transform = compute_train_transform(seed=2147483647)
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=2, shuffle=False, num_workers=2)
    dataiter = iter(trainloader)
    images, labels = next(dataiter)
    img = torchvision.utils.make_grid(images)
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()
    output = images

    print("Maximum error in data augmentation: %g"%rel_error( output.numpy(), correct_output.numpy()))

# Should be 小于 1e-07.
test_data_augmentation(answers['data_augmentation'])

# Base Encoder 和 Projection Head
接下来，把 base encoder 和 projection head 应用于增强样本 **$\tilde{x}_i$** 和 **$\tilde{x}_j$**。

base encoder $f$ 会为增强样本提取表示向量。SimCLR 论文发现，使用更深、更宽的模型可以提升性能，因此选择 [ResNet](https://arxiv.org/pdf/1512.03385.pdf) 作为 base encoder。base encoder 的输出是表示向量 **$h_i = f(\tilde{x}_i$)** 和 **$h_j = f(\tilde{x}_j$)**。

projection head $g$ 是一个小型神经网络，它把表示向量 **$h_i$** 和 **$h_j$** 映射到应用对比损失的空间。论文发现，使用非线性 projection head 可以提升其前一层的表示质量。具体来说，他们使用一个含一层隐藏层的 MLP 作为 projection head $g$。对比损失随后基于输出 **$z_i = g(h_i$)** 和 **$z_j = g(h_j$)** 计算。

我们在 `cs231n/simclr/model.py` 中提供了这两部分的实现。请快速阅读该文件，确保你理解实现。

# SimCLR：对比损失

一个包含 $N$ 张训练图像的 mini-batch 会产生总共 $2N$ 个数据增强样本。对每个增强样本正样本对 $(i, j)$，对比损失函数的目标是最大化向量 $z_i$ 和 $z_j$ 的一致性。具体来说，该损失是 normalized temperature-scaled cross entropy loss，目标是相对于 batch 中所有其他增强样本，最大化 $z_i$ 和 $z_j$ 的一致性：

$$
l \; (i, j) = -\log \frac{\exp (\;\text{sim}(z_i, z_j)\; / \;\tau) }{\sum_{k=1}^{2N} \mathbb{1}_{k \neq i} \exp (\;\text{sim} (z_i, z_k) \;/ \;\tau) }
$$

其中 $\mathbb{1} \in \{0, 1\}$ 是指示函数，当 $k\neq i$ 时输出 1，否则输出 0。$\tau$ 是 temperature 参数，决定指数项增长速度。

sim$(z_i, z_j) = \frac{z_i \cdot z_j}{|| z_i || || z_j ||}$ 是向量 $z_i$ 和 $z_j$ 之间的归一化点积。$z_i$ 和 $z_j$ 越相似，点积越大，分子也越大。分母通过对 batch 中 $z_i$ 和所有其他增强样本 $k$ 求和来归一化该值。归一化值范围为 $(0, 1)$；接近 1 的高分表示正样本对 $(i, j)$ 相似度高，同时 $i$ 与 batch 中其他增强样本 $k$ 相似度低。取负对数后，会把 $(0, 1)$ 范围映射到损失值。

总损失在 batch 中所有正样本对 $(i, j)$ 上计算。令 $z = [z_1, z_2, ..., z_{2N}]$ 包含 batch 中所有增强样本，其中 $z_{1}...z_{N}$ 是左分支输出，$z_{N+1}...z_{2N}$ 是右分支输出。因此，正样本对为 $(z_{k}, z_{k + N})$，对所有 $k \in [1, N]$ 成立。

于是总损失 $L$ 为：

$$
L = \frac{1}{2N} \sum_{k=1}^N [ \; l(k, \;k+N) + l(k+N, \;k)\;]
$$

**注意：** 这个公式与论文中的公式略有不同。我们重新排列了 batch 中正样本对的顺序，所以索引不同。这个重新排列会让向量化实现更容易。

我们会逐步实现向量化形式的损失函数。请在 `cs231n/simclr/contrastive_loss.py` 中实现 `sim` 和 `simclr_loss_naive` 函数。运行下面的 sanity check 测试你的代码。

In [ ]:
from cs231n.simclr.contrastive_loss import *
answers = torch.load('simclr_sanity_check.key')

In [ ]:
def test_sim(left_vec, right_vec, correct_output):
    output = sim(left_vec, right_vec).cpu().numpy()
    print("Maximum error in sim: %g"%rel_error(correct_output.numpy(), output))

# Should be 小于 1e-07.
test_sim(answers['left'][0], answers['right'][0], answers['sim'][0])
test_sim(answers['left'][1], answers['right'][1], answers['sim'][1])

In [ ]:
def test_loss_naive(left, right, tau, correct_output):
    naive_loss = simclr_loss_naive(left, right, tau).item()
    print("Maximum error in simclr_loss_naive: %g"%rel_error(correct_output, naive_loss))

# Should be 小于 1e-07.
test_loss_naive(answers['left'], answers['right'], 5.0, answers['loss']['5.0'])
test_loss_naive(answers['left'], answers['right'], 1.0, answers['loss']['1.0'])

现在实现向量化版本：在 `cs231n/simclr/contrastive_loss.py` 中实现 `sim_positive_pairs`、`compute_sim_matrix` 和 `simclr_loss_vectorized`。运行下面的 sanity check 测试你的代码。

In [ ]:
def test_sim_positive_pairs(left, right, correct_output):
    sim_pair = sim_positive_pairs(left, right).cpu().numpy()
    print("Maximum error in sim_positive_pairs: %g"%rel_error(correct_output.numpy(), sim_pair))

# Should be 小于 1e-07.
test_sim_positive_pairs(answers['left'], answers['right'], answers['sim'])

In [ ]:
def test_sim_matrix(left, right, correct_output):
    out = torch.cat([left, right], dim=0)
    sim_matrix = compute_sim_matrix(out).cpu()
    assert torch.isclose(sim_matrix, correct_output).all(), "correct: {}. got: {}".format(correct_output, sim_matrix)
    print("Test passed!")

test_sim_matrix(answers['left'], answers['right'], answers['sim_matrix'])

In [ ]:
def test_loss_vectorized(left, right, tau, correct_output):
    vec_loss = simclr_loss_vectorized(left, right, tau, device=left.device).item()
    print("Maximum error in loss_vectorized: %g"%rel_error(correct_output, vec_loss))

# Should be 小于 1e-07.
test_loss_vectorized(answers['left'], answers['right'], 5.0, answers['loss']['5.0'])
test_loss_vectorized(answers['left'], answers['right'], 1.0, answers['loss']['1.0'])

# 实现 train 函数
完成 `cs231n/simclr/utils.py` 中的 `train()` 函数，获取模型输出，并使用 `simclr_loss_vectorized` 计算损失。（请查看 `cs231n/simclr/model.py` 中的 `Model` 类，理解模型流水线和返回值。）

In [ ]:
from cs231n.simclr.data_utils import *
from cs231n.simclr.model import *
from cs231n.simclr.utils import *

### 训练 SimCLR 模型

运行下面的单元加载预训练权重，并继续训练一小段时间。这部分大约需要 10 分钟，并会输出到 `pretrained_model/trained_simclr_model.pth`。

**注意：** 如果看到类似 `_[WARN] Cannot find rule for ..._` 的日志，不用担心。这些与 notebook 中使用的另一个模块有关。你可以通过我们提供的提示和注释验证代码改动是否正确。

In [ ]:
# 不要修改这个单元。
feature_dim = 128
temperature = 0.5
k = 200
batch_size = 64
epochs = 1
temperature = 0.5
percentage = 0.5
pretrained_path = './pretrained_model/pretrained_simclr_model.pth'

# 准备数据。
train_transform = compute_train_transform()
train_data = CIFAR10Pair(root='data', train=True, transform=train_transform, download=True)
train_data = torch.utils.data.Subset(train_data, list(np.arange(int(len(train_data)*percentage))))
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=16, pin_memory=True, drop_last=True)
test_transform = compute_test_transform()
memory_data = CIFAR10Pair(root='data', train=True, transform=test_transform, download=True)
memory_loader = DataLoader(memory_data, batch_size=batch_size, shuffle=False, num_workers=16, pin_memory=True)
test_data = CIFAR10Pair(root='data', train=False, transform=test_transform, download=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=16, pin_memory=True)

# 设置模型和 optimizer config。
model = Model(feature_dim)
model.load_state_dict(torch.load(pretrained_path, map_location='cpu'), strict=False)
model = model.to(device)
flops, params = profile(model, inputs=(torch.randn(1, 3, 32, 32).to(device),))
flops, params = clever_format([flops, params])
print('# Model Params: {} FLOPs: {}'.format(params, flops))
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-6)
c = len(memory_data.classes)

# Training loop.
results = {'train_loss': [], 'test_acc@1': [], 'test_acc@5': []} #<< -- 输出

if not os.path.exists('results'):
    os.mkdir('results')
best_acc = 0.0
for epoch in range(1, epochs + 1):
    train_loss = train(model, train_loader, optimizer, epoch, epochs, batch_size=batch_size, temperature=temperature, device=device)
    results['train_loss'].append(train_loss)
    test_acc_1, test_acc_5 = test(model, memory_loader, test_loader, epoch, epochs, c, k=k, temperature=temperature, device=device)
    results['test_acc@1'].append(test_acc_1)
    results['test_acc@5'].append(test_acc_5)

    # Save 统计量.
    if test_acc_1 > best_acc:
        best_acc = test_acc_1
        torch.save(model.state_dict(), './pretrained_model/trained_simclr_model.pth')

# 微调线性层做分类

现在是时候检验表示向量了。

我们从 SimCLR 模型中移除 projection head，并接上一个线性层，用于微调一个简单分类任务。线性层之前的所有层都被冻结，只训练最终线性层的权重。我们会比较 SimCLR + finetuning 模型和 baseline 模型的表现；baseline 没有提前做自监督学习，模型中所有权重都会被训练。你将亲自看到自监督学习的力量，以及学到的表示向量如何提升下游任务性能。

## Baseline：不使用自监督学习
首先看看 baseline 模型。我们从 SimCLR 模型中移除 projection head，并接上一个线性层，用于微调简单分类任务。这里没有提前做自监督学习，模型中所有权重都会被训练。运行下面的单元。

**注意：** 如果看到较低但合理的性能，不用担心。

In [ ]:
class Classifier(nn.Module):
    def __init__(self, num_class):
        super(Classifier, self).__init__()

        # Encoder.
        self.f = Model().f

        # Classifier.
        self.fc = nn.Linear(2048, num_class, bias=True)

    def forward(self, x):
        x = self.f(x)
        feature = torch.flatten(x, start_dim=1)
        out = self.fc(feature)
        return out

In [ ]:
# 不要修改这个单元。
feature_dim = 128
temperature = 0.5
k = 200
batch_size = 128
epochs = 10
percentage = 0.1

train_transform = compute_train_transform()
train_data = CIFAR10(root='data', train=True, transform=train_transform, download=True)
trainset = torch.utils.data.Subset(train_data, list(np.arange(int(len(train_data)*percentage))))
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=16, pin_memory=True)
test_transform = compute_test_transform()
test_data = CIFAR10(root='data', train=False, transform=test_transform, download=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=16, pin_memory=True)

model = Classifier(num_class=len(train_data.classes)).to(device)
for param in model.f.parameters():
    param.requires_grad = False

flops, params = profile(model, inputs=(torch.randn(1, 3, 32, 32).to(device),))
flops, params = clever_format([flops, params])
print('# Model Params: {} FLOPs: {}'.format(params, flops))
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3, weight_decay=1e-6)
no_pretrain_results = {'train_loss': [], 'train_acc@1': [], 'train_acc@5': [],
           'test_loss': [], 'test_acc@1': [], 'test_acc@5': []}

best_acc = 0.0
for epoch in range(1, epochs + 1):
    train_loss, train_acc_1, train_acc_5 = train_val(model, train_loader, optimizer, epoch, epochs, device='cuda')
    no_pretrain_results['train_loss'].append(train_loss)
    no_pretrain_results['train_acc@1'].append(train_acc_1)
    no_pretrain_results['train_acc@5'].append(train_acc_5)
    test_loss, test_acc_1, test_acc_5 = train_val(model, test_loader, None, epoch, epochs)
    no_pretrain_results['test_loss'].append(test_loss)
    no_pretrain_results['test_acc@1'].append(test_acc_1)
    no_pretrain_results['test_acc@5'].append(test_acc_5)
    if test_acc_1 > best_acc:
        best_acc = test_acc_1

# Print best 测试准确率.
print('Best top-1 accuracy without self-supervised learning: ', best_acc)

## 使用自监督学习

现在看看自监督学习能带来多少提升。这里我们使用你写的 simclr loss 预训练 SimCLR 模型，从模型中移除 projection head，然后使用线性层微调简单分类任务。

In [ ]:
# 不要修改这个单元。
feature_dim = 128
temperature = 0.5
k = 200
batch_size = 128
epochs = 10
percentage = 0.1
pretrained_path = './pretrained_model/trained_simclr_model.pth'

train_transform = compute_train_transform()
train_data = CIFAR10(root='data', train=True, transform=train_transform, download=True)
trainset = torch.utils.data.Subset(train_data, list(np.arange(int(len(train_data)*percentage))))
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=16, pin_memory=True)
test_transform = compute_test_transform()
test_data = CIFAR10(root='data', train=False, transform=test_transform, download=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=16, pin_memory=True)

model = Classifier(num_class=len(train_data.classes))
model.load_state_dict(torch.load(pretrained_path, map_location='cpu'), strict=False)
model = model.to(device)
for param in model.f.parameters():
    param.requires_grad = False

flops, params = profile(model, inputs=(torch.randn(1, 3, 32, 32).to(device),))
flops, params = clever_format([flops, params])
print('# Model Params: {} FLOPs: {}'.format(params, flops))
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3, weight_decay=1e-6)
pretrain_results = {'train_loss': [], 'train_acc@1': [], 'train_acc@5': [],
           'test_loss': [], 'test_acc@1': [], 'test_acc@5': []}

best_acc = 0.0
for epoch in range(1, epochs + 1):
    train_loss, train_acc_1, train_acc_5 = train_val(model, train_loader, optimizer, epoch, epochs)
    pretrain_results['train_loss'].append(train_loss)
    pretrain_results['train_acc@1'].append(train_acc_1)
    pretrain_results['train_acc@5'].append(train_acc_5)
    test_loss, test_acc_1, test_acc_5 = train_val(model, test_loader, None, epoch, epochs)
    pretrain_results['test_loss'].append(test_loss)
    pretrain_results['test_acc@1'].append(test_acc_1)
    pretrain_results['test_acc@5'].append(test_acc_5)
    if test_acc_1 > best_acc:
        best_acc = test_acc_1

# 打印最佳测试准确率。你应该看到 best top-1 accuracy >= 70%。
print('Best top-1 accuracy with self-supervised learning: ', best_acc)

### 绘制对比

绘制 baseline 模型（无预训练）和同一模型经过自监督学习预训练后的测试准确率对比。

In [ ]:
plt.plot(no_pretrain_results['test_acc@1'], label="Without Pretrain")
plt.plot(pretrain_results['test_acc@1'], label="With Pretrain")
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Test Top-1 Accuracy')
plt.legend()
plt.show()